# Link gold queries with fused search + produce the dense/sparse/fused ablation table

Supersedes `link_gold_queries.ipynb`. That notebook used dense-only top-1 to link
the 280 `Gold_KB_final_280` questions to a `correct_answer_id`, and found a
52% category-mismatch rate (146/280) -- real signal, not noise, but dense-only
search is the weakest configuration available (no sparse leg, no rerank), so
some of that is exactly what Lever 3 exists to fix.

This notebook:
1. Runs **dense**, **sparse**, and **fused (RRF)** search for all 280 questions
2. Uses **fused's top-1** as the ground-truth `correct_answer_id` (better
   matching accuracy than dense-alone -- still flagged for human review, not
   silently trusted)
3. Measures Recall@1/5/20 for dense-only and sparse-only **against that fused
   ground truth** -- this is the Lever 3 ablation table, and it reuses the
   dense-only linking work as one leg of it rather than throwing it away

**Caveat, stated plainly:** fused's own "recall" against a ground truth defined
*by its own top-1* is close to tautological -- of course fused finds what fused
said was correct. The meaningful numbers here are how often dense-only and
sparse-only *independently* land on the same answer fused converged on. Once a
person has reviewed the flagged rows (category mismatch or low confidence),
rerun the Recall cells against the corrected `correct_answer_id` column for a
cleaner number.

Needs the same Qdrant credentials as before, and **Runtime -> Change runtime
type -> T4 GPU**.

In [ ]:
!pip install -q qdrant-client FlagEmbedding

In [ ]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd
from retrieval.search import HybridRetriever, reciprocal_rank_fusion

print("cloned + imported OK")

In [ ]:
import csv

with open("eval/gold_kb_final_280_raw.csv", encoding="utf-8-sig", newline="") as f:
    gold_rows = list(csv.DictReader(f))

with open("knowledge_base/Womens_Health_KB - 2000_final.csv", encoding="utf-8", newline="") as f:
    kb_rows = list(csv.DictReader(f))

sindhi_cats_order = []
seen_cats = set()
for r in kb_rows:
    if r["category"] not in seen_cats:
        sindhi_cats_order.append(r["category"]); seen_cats.add(r["category"])

EN_CATEGORY_ORDER = [
    "Menstrual Health & Periods", "Mental Health & Emotional Well-being",
    "Pregnancy & Maternal Health", "PCOS & Hormonal Health",
    "Women's Nutrition & Wellness", "Menopause & Menopausal Health",
    "Fertility & Reproductive Health", "Vaginal & Personal Hygiene",
]
sd_to_en_category = dict(zip(sindhi_cats_order, EN_CATEGORY_ORDER))

print(f"{len(gold_rows)} gold rows, {len(kb_rows)} KB rows, category map built")

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

In [ ]:
from getpass import getpass
from qdrant_client import QdrantClient

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def embed_fn(text):
    normalized = normalize_sd(text)
    out = model.encode([normalized], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    return {"dense": out["dense_vecs"][0].tolist(), "sparse": out["lexical_weights"][0]}

retriever = HybridRetriever(client, embed_fn=embed_fn)
print("retriever ready, collection points:", client.get_collection("naari_ai_kb").points_count)

In [ ]:
TOP_K = 20
linked = []

for i, row in enumerate(gold_rows, start=1):
    query = row["Question"]

    dense_rows = retriever.dense_search(query, top_k=TOP_K)
    sparse_rows = retriever.sparse_search(query, top_k=TOP_K)

    dense_ids = [r["answer_id"] for r in dense_rows]
    sparse_ids = [r["answer_id"] for r in sparse_rows]
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids])
    fused_ids = [aid for aid, _score in fused]

    row_by_id = {r["answer_id"]: r for r in dense_rows + sparse_rows}
    top1_id = fused_ids[0] if fused_ids else None
    top1_row = row_by_id.get(top1_id)
    top1_cat_en = sd_to_en_category.get(top1_row["category"], top1_row["category"]) if top1_row else ""

    linked.append({
        "query_id": f"gold_{row['ID']}",
        "query": query,
        "stated_category": row["Category"],
        "stated_subcategory": row["Subcategory"],
        "correct_answer_id": top1_id,
        "fused_top1_category_en": top1_cat_en,
        "category_match": (top1_cat_en == row["Category"]),
        "dense_top1_id": dense_ids[0] if dense_ids else None,
        "dense_top1_score": f"{dense_rows[0]['score']:.4f}" if dense_rows else "",
        "sparse_top1_id": sparse_ids[0] if sparse_ids else None,
        "sparse_top1_score": f"{sparse_rows[0]['score']:.4f}" if sparse_rows else "",
        "low_confidence": (dense_rows[0]["score"] < 0.6) if dense_rows else True,
        "gold_own_answer": row["Answer"],
        "gold_own_source": row["Source"],
        "_dense_ids": dense_ids,
        "_sparse_ids": sparse_ids,
        "_fused_ids": fused_ids,
    })

    if i % 40 == 0:
        print(f"linked {i}/{len(gold_rows)}")

print(f"done, {len(linked)} rows linked")

In [ ]:
import csv

OUT_PATH = "eval/gold_eval_280_linked.csv"
public_fields = [k for k in linked[0].keys() if not k.startswith("_")]
with open(OUT_PATH, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=public_fields)
    w.writeheader()
    for r in linked:
        w.writerow({k: r[k] for k in public_fields})

mismatches = [r for r in linked if not r["category_match"]]
low_conf = [r for r in linked if r["low_confidence"]]
print(f"wrote {OUT_PATH}")
print(f"category mismatches (fused top-1 vs stated category): {len(mismatches)}/{len(linked)}")
print(f"low-confidence dense matches (<0.6): {len(low_conf)}/{len(linked)}")
print("-> review these before treating correct_answer_id as final ground truth.")

## Ablation table: dense-only vs sparse-only vs fused, Recall@1/5/20

In [ ]:
def recall_at_k(ground_truth_key, ranked_key, k):
    hits = 0
    for r in linked:
        gt = r[ground_truth_key]
        ranked = r[ranked_key][:k]
        if gt in ranked:
            hits += 1
    return hits / len(linked)

results = {}
for leg, key in [("dense", "_dense_ids"), ("sparse", "_sparse_ids"), ("fused", "_fused_ids")]:
    results[leg] = {f"recall@{k}": recall_at_k("correct_answer_id", key, k) for k in (1, 5, 20)}

for leg, metrics in results.items():
    print(leg, metrics)

In [ ]:
import datetime

lines = []
lines.append("# Phase 1 dense/sparse/fused ablation -- gold_eval_280 (PROVISIONAL)\n")
lines.append(f"Generated: {datetime.datetime.utcnow().isoformat()}Z\n")
lines.append("\n")
lines.append("**Ground truth = fused search's own top-1 answer**, not independently "
             "human-verified yet. Fused's row is close to tautological (it is being "
             "measured against itself) -- the meaningful comparison is dense-only and "
             "sparse-only against that same reference point. "
             f"{len(mismatches)}/280 rows are flagged for category mismatch and "
             f"{len(low_conf)}/280 for low confidence; re-run after human review for a "
             "trustworthy final number.\n\n")
lines.append("| Leg | Recall@1 | Recall@5 | Recall@20 |\n")
lines.append("|---|---:|---:|---:|\n")
for leg in ("dense", "sparse", "fused"):
    m = results[leg]
    lines.append(f"| {leg} | {m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@20']:.3f} |\n")

with open("eval/results.md", "a", encoding="utf-8") as f:
    f.write("\n\n" + "".join(lines))

print("appended to eval/results.md")
print("".join(lines))

In [ ]:
from google.colab import files
files.download("eval/gold_eval_280_linked.csv")
files.download("eval/results.md")

## Next steps

1. Someone reviews every row flagged `category_match = False` or `low_confidence = True` in `eval/gold_eval_280_linked.csv`, correcting or dropping `correct_answer_id` as needed.
2. Re-run the Recall@K cell against the corrected column for a trustworthy final ablation table -- the one just written to `eval/results.md` is explicitly provisional.
3. Still confirm separately with whoever built `Gold_KB_final_280.csv` whether these 280 questions were drawn from real harvested speech -- linking to an answer_id doesn't resolve that.